# Pipeline complet `bottle` — Prétraitement + Autoencodeur convolutionnel (détection d'anomalies, MVTec-AD)

Ce notebook implémente le pipeline décrit dans `preprocessing_bottle_fr.md`, en deux parties :

**Partie 1 — Prétraitement** (étapes 1 à 6, dont 3.1 à 3.11) :
1. Inventaire et vérification des fichiers (images + masques).
2. Redimensionnement des images (128×128 ou 256×256) et des masques (`ground_truth`).
3. Normalisation des pixels dans `[0, 1]`.
4. Split train/validation sur les images saines uniquement.
5. Augmentation (Albumentations) appliquée uniquement à l'entraînement.
6. Pipelines `tf.data` pour train/val/test, étude de déséquilibre de classe.

**Partie 2 — Autoencodeur convolutionnel** (étapes 7 à 15) :
7. Conception de l'architecture (encodeur / bottleneck / décodeur).
8. Construction et lecture du `summary()` (incl. ratio de compression).
9. Vérification sur un vrai batch du pipeline.
10. Compilation (`Adam`, loss `MSE`, métrique `SSIM`).
11. Suivi MLflow (params, métriques par epoch, artifacts).
12. Entraînement (cible = entrée, reconstruction).
13. Courbes d'apprentissage (loss + SSIM, train/validation).
14. Exemples de reconstruction après entraînement.
15. Prochaine étape (score d'anomalie).

**Bibliothèques** : `pillow` (I/O), `opencv` (resize images), `scikit-image` (resize masques binaires), `albumentations` (augmentation), `tensorflow`/`keras` (pipeline + modèle), `mlflow` (suivi des runs).

> Adapter `DATA_DIR` au chemin réel de ton dataset `bottle` avant exécution.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # réduit les logs TensorFlow

from pathlib import Path
import numpy as np
import cv2
from PIL import Image
from skimage.transform import resize as sk_resize
import albumentations as A
import tensorflow as tf
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## Configuration (correspond à la section 1 du md — choix de la taille cible)

In [ ]:
DATA_DIR = Path("data/bottle")    # <- à adapter à ton chemin local
IMG_SIZE = 128                    # 128 pour itérer vite, 256 pour la version finale
BATCH_SIZE = 16
VAL_FRACTION = 0.30               # initial 0.15, tested with 0.25
SEED = 42

np.random.seed(SEED)

## Étape 3.1 — Inventaire et vérification des fichiers

- Liste des images saines (`train/good`, `test/good`) et des dossiers de défauts (`test/<defect>`).
- Vérification que chaque image défectueuse a bien son masque correspondant dans `ground_truth/<defect>`.
- Détection de fichiers corrompus.

In [ ]:
train_good = sorted((DATA_DIR / "train" / "good").glob("*.png"))
test_good = sorted((DATA_DIR / "test" / "good").glob("*.png"))
defect_dirs = [d for d in (DATA_DIR / "test").iterdir() if d.is_dir() and d.name != "good"]
defect_names = sorted(d.name for d in defect_dirs)

print("train/good :", len(train_good))
print("test/good  :", len(test_good))
print("classes de défauts :", defect_names)

In [ ]:
# Vérifie la correspondance image <-> masque pour chaque classe de défaut
# Convention MVTec-AD : ground_truth/<defect>/<id>_mask.png <-> test/<defect>/<id>.png
for name in defect_names:
    imgs = sorted((DATA_DIR / "test" / name).glob("*.png"))
    masks = sorted((DATA_DIR / "ground_truth" / name).glob("*_mask.png"))
    assert len(imgs) == len(masks), f"Mismatch images/masks pour {name}: {len(imgs)} vs {len(masks)}"
    print(f"  {name}: {len(imgs)} images / {len(masks)} masks — OK")

In [ ]:
def verify_images(paths):
    """Retourne la liste des fichiers illisibles/corrompus."""
    bad = []
    for p in paths:
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception:
            bad.append(p)
    return bad

bad_files = verify_images(train_good + test_good)
print("Fichiers corrompus détectés :", len(bad_files))
if bad_files:
    print(bad_files)

## Étapes 3.2 à 3.4 — Chargement, redimensionnement (images + masques) et normalisation

- **Images** : Pillow pour la lecture (conversion RGB systématique), OpenCV pour le resize (`INTER_AREA`, adapté à la réduction de taille).
- **Masques** : lecture Pillow en niveaux de gris, resize scikit-image en **plus proche voisin** (`order=0`) pour ne pas casser la binarité, puis re-binarisation explicite.

In [ ]:
def load_and_resize_image(path, size=IMG_SIZE):
    """Charge une image et la redimensionne en (size, size, 3), RGB, uint8."""
    img = Image.open(path).convert("RGB")
    img = np.array(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img


def load_and_resize_mask(path, size=IMG_SIZE):
    """Charge un masque binaire et le redimensionne sans casser la binarité."""
    mask = Image.open(path).convert("L")
    mask = np.array(mask)
    mask = sk_resize(mask, (size, size), order=0, preserve_range=True, anti_aliasing=False)
    mask = (mask > 127).astype(np.uint8)
    return mask


def normalize(img):
    """Normalise une image uint8 [0,255] en float32 [0,1]."""
    return img.astype(np.float32) / 255.0

In [ ]:
# Vérification rapide sur un exemple
sample_img = load_and_resize_image(train_good[0])
print("Image redimensionnée :", sample_img.shape, sample_img.dtype)

if defect_names:
    sample_mask_path = sorted((DATA_DIR / "ground_truth" / defect_names[0]).glob("*_mask.png"))[0]
    sample_mask = load_and_resize_mask(sample_mask_path)
    print("Masque redimensionné :", sample_mask.shape, "valeurs uniques :", np.unique(sample_mask))

sample_norm = normalize(sample_img)
print("Image normalisée, plage :", sample_norm.min(), "-", sample_norm.max())

In [ ]:
# Aperçu visuel : image originale vs redimensionnée
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(Image.open(train_good[0]).convert("RGB"))
axes[0].set_title("Originale")
axes[0].axis("off")
axes[1].imshow(sample_img)
axes[1].set_title(f"Redimensionnée {IMG_SIZE}x{IMG_SIZE}")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## Étape 3.5 — Split train / validation (images saines uniquement)

Split aléatoire sur les chemins de fichiers de `train/good`, avant tout chargement en mémoire. Les dossiers `test/` (sains et défectueux) ne sont jamais utilisés pour ce split — ils servent uniquement à l'évaluation finale.

In [ ]:
train_paths, val_paths = train_test_split(
    train_good, test_size=VAL_FRACTION, random_state=SEED
)
print(f"Train : {len(train_paths)} images")
print(f"Validation : {len(val_paths)} images")

## Étape 3.6 — Pipeline d'augmentation (Albumentations) — images saines d'entraînement uniquement

Transformations choisies, cohérentes avec un contexte industriel (une bouteille doit rester reconnaissable comme "normale") :
- `HorizontalFlip` — symétrie horizontale
- `Rotate` — rotation légère (±15°)
- `ShiftScaleRotate` — translation + mise à l'échelle légère (centrage/zoom imparfait de la caméra)
- `RandomBrightnessContrast` — variations d'éclairage

Appliquées uniquement sur le split d'entraînement (`train_paths`), jamais sur validation/test.

In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_REPLICATE),
    A.ShiftScaleRotate(
        shift_limit=0.05, scale_limit=0.1, rotate_limit=0,
        border_mode=cv2.BORDER_REPLICATE, p=0.5,
    ),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
])

In [ ]:
# Aperçu rapide de quelques tirages d'augmentation sur une même image
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
base_img = load_and_resize_image(train_paths[0])
axes[0].imshow(base_img); axes[0].set_title("Original"); axes[0].axis("off")
for i in range(1, 4):
    aug_img = train_transform(image=base_img)["image"]
    axes[i].imshow(aug_img); axes[i].set_title(f"Augmentation {i}"); axes[i].axis("off")
plt.tight_layout()
plt.show()

## Étape 3.7 — Vérification visuelle de l'augmentation (original vs augmenté)

Pour confirmer que le pipeline d'augmentation produit des transformations réalistes (et qu'il n'est pas un no-op), on compare, pour plusieurs images saines différentes :
- l'image **originale** redimensionnée (ligne du haut)
- **plusieurs tirages augmentés** de la même image (lignes suivantes — chaque appel du pipeline étant stochastique)

In [ ]:
N_IMAGES = 4      # nombre d'images saines comparées
N_DRAWS = 3        # nombre de tirages augmentés par image

fig, axes = plt.subplots(N_DRAWS + 1, N_IMAGES, figsize=(3.2 * N_IMAGES, 3.2 * (N_DRAWS + 1)))

for col in range(N_IMAGES):
    path = train_paths[col]
    original = load_and_resize_image(path)

    axes[0, col].imshow(original)
    axes[0, col].set_title(f"Original\n{path.name}")
    axes[0, col].axis("off")

    for row in range(1, N_DRAWS + 1):
        augmented = train_transform(image=original)["image"]
        axes[row, col].imshow(augmented)
        axes[row, col].set_title(f"Augmenté #{row}")
        axes[row, col].axis("off")

plt.suptitle("Comparaison original (haut) vs tirages augmentés (bas)")
plt.tight_layout()
plt.show()

## Étape 3.8 — Vérification visuelle du chargement

Avant de construire le pipeline `tf.data`, on affiche une grille d'images chargées/redimensionnées :
- quelques images **saines** (`train/good`)
- quelques images de **chaque défaut** (`test/<defect>`), avec leur masque `ground_truth` superposé

Objectif : détecter à l'œil un souci de chargement (inversion BGR/RGB, mauvais alignement image/masque, resize qui écrase l'information) avant d'aller plus loin.

In [ ]:
N_SAMPLES = 4

fig, axes = plt.subplots(2, N_SAMPLES, figsize=(4 * N_SAMPLES, 8))

# Ligne 1 : images saines
for i, path in enumerate(train_good[:N_SAMPLES]):
    img = load_and_resize_image(path)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"good\n{path.name}")
    axes[0, i].axis("off")

# Ligne 2 : un défaut par colonne (image + masque superposé)
n_defects_shown = min(N_SAMPLES, len(defect_names))
for i in range(N_SAMPLES):
    axes[1, i].axis("off")
for i, name in enumerate(defect_names[:n_defects_shown]):
    img_path = sorted((DATA_DIR / "test" / name).glob("*.png"))[0]
    mask_path = sorted((DATA_DIR / "ground_truth" / name).glob("*_mask.png"))[0]
    img = load_and_resize_image(img_path)
    mask = load_and_resize_mask(mask_path)

    axes[1, i].imshow(img)
    axes[1, i].imshow(mask, cmap="Reds", alpha=0.4)  # masque en surimpression
    axes[1, i].set_title(f"{name}\n{img_path.name}")
    axes[1, i].axis("off")

plt.suptitle("Vérification visuelle : saines (haut) vs défauts + masque superposé (bas)")
plt.tight_layout()
plt.show()

## Étape 3.9 — Construction du pipeline `tf.data` (entraînement / validation)

- `load_and_resize_image` (Pillow/OpenCV) et Albumentations ne sont pas nativement compatibles avec le graphe TensorFlow → encapsulation via `tf.py_function`.
- Augmentation appliquée uniquement si `augment=True` (train), jamais pour validation/test.
- `.cache()` optionnel si le dataset tient en mémoire (cas de `bottle`, quelques centaines d'images).

In [ ]:
def make_loader(augment: bool):
    def _load(path):
        path = path.numpy().decode("utf-8")
        img = load_and_resize_image(path)
        if augment:
            img = train_transform(image=img)["image"]
        img = normalize(img)
        return img.astype(np.float32)

    def _tf_wrapper(path):
        img = tf.py_function(_load, [path], tf.float32)
        img.set_shape((IMG_SIZE, IMG_SIZE, 3))
        return img
    return _tf_wrapper


def build_dataset(paths, augment=False, shuffle=False, batch_size=BATCH_SIZE, cache=True):
    paths_str = [str(p) for p in paths]
    ds = tf.data.Dataset.from_tensor_slices(paths_str)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths_str), seed=SEED)
    ds = ds.map(make_loader(augment), num_parallel_calls=tf.data.AUTOTUNE)
    if cache and not augment:
        # pas de cache si augmentation (on veut une augmentation différente à chaque epoch)
        ds = ds.cache()
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
train_ds = build_dataset(train_paths, augment=True, shuffle=True)
val_ds = build_dataset(val_paths, augment=False, shuffle=False)

for batch in train_ds.take(1):
    print("Batch shape :", batch.shape)
    print("dtype :", batch.dtype)
    print("Plage de valeurs :", float(tf.reduce_min(batch)), "-", float(tf.reduce_max(batch)))

## Étape 3.10 — Construction des datasets de test (évaluation)

- `test/good` + chaque `test/<defect>` : **pas d'augmentation**, juste resize + normalisation.
- Masques `ground_truth/<defect>` redimensionnés en parallèle pour l'évaluation pixel-level future (IoU, AUC-ROC pixel).

In [ ]:
test_good_ds = build_dataset(test_good, augment=False, shuffle=False, cache=False)

test_defect_ds = {}
test_defect_masks = {}
for name in defect_names:
    imgs = sorted((DATA_DIR / "test" / name).glob("*.png"))
    masks = sorted((DATA_DIR / "ground_truth" / name).glob("*_mask.png"))
    test_defect_ds[name] = build_dataset(imgs, augment=False, shuffle=False, cache=False)
    test_defect_masks[name] = np.stack([load_and_resize_mask(m) for m in masks])
    print(f"{name}: dataset images prêt, masques shape={test_defect_masks[name].shape}")

## Étape 3.11 — Étude de déséquilibre de classe

Deux niveaux à étudier, tous les deux dans `test/` (`train/good` n'a qu'une seule classe) :
- **Niveau image** : proportion saines vs défectueuses, et répartition entre types de défauts.
- **Niveau pixel** : à l'intérieur de chaque masque, fraction de pixels "défaut" vs "fond" — pertinent seulement si un modèle de segmentation est envisagé.

In [ ]:
# --- Niveau image : comptage par classe ---
n_good_test = len(test_good)
n_defect_total = sum(len(v) for v in test_defect_masks.values())

print("train/good:", len(train_good))
print("test/good:", n_good_test)
for name in defect_names:
    print(f"test/{name}:", test_defect_masks[name].shape[0])

print(f"\nRatio saines/défectueuses (test) : {n_good_test}/{n_good_test + n_defect_total}"
      f" ({n_good_test/(n_good_test+n_defect_total):.1%}) vs "
      f"{n_defect_total}/{n_good_test + n_defect_total} ({n_defect_total/(n_good_test+n_defect_total):.1%})")

In [ ]:
# --- Niveau pixel : fraction de pixels défectueux par masque (résolution originale) ---
pixel_fractions = {}
for name in defect_names:
    mask_paths = sorted((DATA_DIR / "ground_truth" / name).glob("*_mask.png"))
    fracs = []
    for p in mask_paths:
        m = np.array(Image.open(p).convert("L"))
        fracs.append((m > 127).sum() / m.size)
    pixel_fractions[name] = np.array(fracs)
    print(f"{name}: frac pixels défectueux — min={fracs and min(fracs):.4f} "
          f"mean={pixel_fractions[name].mean():.4f} max={pixel_fractions[name].max():.4f}")

In [ ]:
# --- Synthèse visuelle : comptage par classe + distribution des fractions de pixels ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

labels = ["train/good", "test/good"] + [f"test/{n}" for n in defect_names]
counts = [len(train_good), n_good_test] + [test_defect_masks[n].shape[0] for n in defect_names]
colors = ["#4C72B0", "#55A868"] + ["#C44E52"] * len(defect_names)
axes[0].bar(labels, counts, color=colors)
axes[0].set_title("Nombre d'images par classe")
axes[0].set_ylabel("Nombre d'images")
axes[0].tick_params(axis="x", rotation=40)
for i, c in enumerate(counts):
    axes[0].text(i, c + 2, str(c), ha="center")

axes[1].boxplot([pixel_fractions[n] for n in defect_names], tick_labels=defect_names)
axes[1].set_title("Fraction de pixels défectueux par masque")
axes[1].set_ylabel("Fraction de pixels défectueux")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## Résumé (correspond aux sections 4 à 6 du md — arborescence, points de vigilance, prochaine étape)

| Étape (md) | Contenu | Statut |
|---|---|---|
| 3.1 | Inventaire + vérification fichiers | ✅ |
| 3.2 | Resize images ({IMG_SIZE}×{IMG_SIZE}) | ✅ |
| 3.3 | Resize masques (binaire préservé) | ✅ |
| 3.4 | Normalisation [0, 1] | ✅ |
| 3.5 | Split train/validation (images saines) | ✅ |
| 3.6 | Pipeline d'augmentation (Albumentations) | ✅ |
| 3.7 | Vérification visuelle de l'augmentation | ✅ |
| 3.8 | Vérification visuelle du chargement | ✅ |
| 3.9 | Pipeline `tf.data` (train/validation) | ✅ |
| 3.10 | Datasets de test (évaluation) | ✅ |
| 3.11 | Étude de déséquilibre de classe | ✅ |

**Prochaine étape** (section 6 du md) : définir l'architecture du modèle (autoencodeur convolutif pour la reconstruction, ou approche par extraction de features + distance type PaDiM) et le protocole d'évaluation (AUC-ROC image-level sur `test/`, IoU/AUC-ROC pixel-level via `ground_truth/`).

# Partie 2 — Autoencodeur convolutionnel (conception, entraînement, suivi MLflow)

Cette partie réutilise directement `train_ds` et `val_ds` construits à l'étape 3.9 (déjà normalisés `[0,1]`, `train_ds` avec augmentation, `val_ds` sans), ainsi que `IMG_SIZE`/`BATCH_SIZE` définis en configuration. Pas de rechargement de données ici.

In [ ]:
from tensorflow.keras import layers, Model
import mlflow

## 7. Conception de l'architecture de l'autoencodeur

Un autoencodeur convolutionnel a deux moitiés symétriques :

- **Encodeur** : réduit progressivement la résolution spatiale (`strides=2`) tout en augmentant le nombre de filtres — échange l'information spatiale contre de l'information sémantique.
- **Bottleneck** : la représentation la plus compacte, au milieu du réseau — c'est la contrainte qui force le modèle à apprendre une structure du "normal" plutôt que de simplement copier l'image.
- **Décodeur** : miroir inverse (`Conv2DTranspose`), remonte en résolution jusqu'à retrouver la taille et le nombre de canaux de l'image d'origine.

**Choix de conception faits ici** :
- `strides=2` plutôt que `MaxPooling2D` : la convolution stridée apprend elle-même comment sous-échantillonner.
- Filtres qui doublent à chaque étage de l'encodeur (32→64→128→256) : compense la perte de résolution spatiale par plus de capacité de représentation.
- `padding="same"` partout : simplifie le calcul des tailles, garantit la symétrie encodeur/décodeur.
- Activation `sigmoid` en sortie : cohérente avec la normalisation `[0,1]` du pipeline (`indusense.vision.dataset.normalize`).

In [ ]:
def build_autoencoder(img_size: int = IMG_SIZE, base_filters: int = 32) -> Model:
    """Autoencodeur convolutionnel symétrique.

    Encodeur : 4 Conv2D stridées (downsampling ×2 à chaque étage).
    Bottleneck : représentation (img_size/16, img_size/16, base_filters*8).
    Décodeur : 4 Conv2DTranspose stridées (upsampling ×2 à chaque étage), miroir exact de l'encodeur.
    """
    inputs = layers.Input(shape=(img_size, img_size, 3), name="input_image")

    # --- Encodeur ---
    x = layers.Conv2D(base_filters,     3, strides=2, padding="same", activation="relu", name="enc_conv1")(inputs)
    x = layers.Conv2D(base_filters * 2, 3, strides=2, padding="same", activation="relu", name="enc_conv2")(x)
    x = layers.Conv2D(base_filters * 4, 3, strides=2, padding="same", activation="relu", name="enc_conv3")(x)
    x = layers.Conv2D(base_filters * 8, 3, strides=2, padding="same", activation="relu", name="enc_conv4")(x)

    latent = x  # bottleneck

    # --- Décodeur ---
    x = layers.Conv2DTranspose(base_filters * 4, 3, strides=2, padding="same", activation="relu", name="dec_conv1")(latent)
    x = layers.Conv2DTranspose(base_filters * 2, 3, strides=2, padding="same", activation="relu", name="dec_conv2")(x)
    x = layers.Conv2DTranspose(base_filters,     3, strides=2, padding="same", activation="relu", name="dec_conv3")(x)
    outputs = layers.Conv2DTranspose(3, 3, strides=2, padding="same", activation="sigmoid", name="dec_output")(x)

    return Model(inputs, outputs, name="conv_autoencoder")


autoencoder = build_autoencoder()

### Informations clés du bottleneck

Avant de lire le `summary()` complet, on isole les informations qui caractérisent le bottleneck lui-même — c'est la partie de l'architecture qui détermine la capacité de compression du modèle, donc sa capacité à distinguer le "normal" de l'anormal.

In [ ]:
bottleneck_layer_name = "enc_conv4"
bottleneck_layer = autoencoder.get_layer(bottleneck_layer_name)
bottleneck_shape = bottleneck_layer.output.shape[1:]   # (H', W', C'), sans la dimension de batch
input_shape = autoencoder.input.shape[1:]              # (H, W, C)

bottleneck_values = int(np.prod(bottleneck_shape))
input_values = int(np.prod(input_shape))
spatial_reduction = input_shape[0] // bottleneck_shape[0]   # ex. 128 // 8 = 16 (÷16 en hauteur/largeur)
compression_ratio = input_values / bottleneck_values

print(f"Couche bottleneck         : {bottleneck_layer_name}")
print(f"Forme entrée              : {tuple(input_shape)}  ({input_values} valeurs)")
print(f"Forme bottleneck          : {tuple(bottleneck_shape)}  ({bottleneck_values} valeurs)")
print(f"Réduction spatiale        : ÷{spatial_reduction} en hauteur et en largeur")
print(f"Canaux bottleneck         : {bottleneck_shape[-1]}")
print(f"Ratio de compression      : {compression_ratio:.2f}x")

## 8. Lecture du `summary()`

**Rappel bottleneck** (calculé en étape 7, affiché ci-dessous en évidence) : c'est `enc_conv4` qui porte le `Output Shape` le plus petit du tableau — repérer cette ligne en premier, c'est elle qui contraint toute la capacité du modèle.

Points à vérifier systématiquement dans le tableau affiché :

1. **Output Shape de la dernière couche == Output Shape de l'entrée** (`(None, 128, 128, 3)` des deux côtés) — condition nécessaire pour comparer la reconstruction à l'original pixel à pixel.
2. **Param # d'une `Conv2D`** : `(kernel_h × kernel_w × canaux_entrée + 1) × canaux_sortie` (le `+1` = biais par filtre).
3. **Symétrie du nombre de paramètres** entre encodeur et décodeur — un déséquilibre marqué signale souvent un bottleneck mal dimensionné.
4. **Total params vs volume de données** : à mettre en regard du nombre d'images d'entraînement pour anticiper un risque de sur-apprentissage.

In [ ]:
print("=" * 60)
print(f"BOTTLENECK ({bottleneck_layer_name}) — à repérer dans le summary() ci-dessous")
print(f"  Forme            : {tuple(bottleneck_shape)}")
print(f"  Compression       : {input_values} -> {bottleneck_values} valeurs ({compression_ratio:.2f}x)")
print("=" * 60)

In [ ]:
autoencoder.summary()

In [ ]:
# Vérification manuelle du calcul de Param # pour les deux premières couches (formule ci-dessus)
def conv2d_params(kernel, in_channels, out_channels):
    return (kernel * kernel * in_channels + 1) * out_channels

print("enc_conv1 attendu :", conv2d_params(3, 3, 32), "  (résultat summary() : 896)")
print("enc_conv2 attendu :", conv2d_params(3, 32, 64), " (résultat summary() : 18496)")

### Ratio de compression

Le bottleneck ne se juge pas qu'au nombre de paramètres — ce qui compte, c'est combien de **valeurs** il faut pour représenter une image une fois passée dans l'encodeur, comparé au nombre de valeurs de l'image d'origine :

```
ratio de compression = (nb de valeurs en entrée) / (nb de valeurs dans le latent)
                      = (H × W × C entrée) / (H' × W' × C' latent)
```

Un ratio élevé (bottleneck très compact) force le modèle à apprendre une représentation plus abstraite du "normal" — utile pour la détection d'anomalie, mais risque de sous-apprentissage (reconstruction dégradée même sur des images saines) si le ratio est trop agressif. Un ratio trop faible (bottleneck presque aussi grand que l'entrée) risque à l'inverse de laisser le modèle apprendre une quasi-identité, peu informative pour distinguer sain/défectueux.

In [ ]:
# Valeurs déjà calculées en étape 7 (bottleneck_shape, bottleneck_values, compression_ratio) — réaffichées ici en contexte
print(f"Entrée   : {tuple(input_shape)} -> {input_values} valeurs")
print(f"Latent   : {tuple(bottleneck_shape)} -> {bottleneck_values} valeurs")
print(f"Ratio de compression : {compression_ratio:.2f}x")

## 9. Vérification sur un vrai batch du pipeline

In [ ]:
for batch in train_ds.take(1):
    reconstruction = autoencoder(batch)
    print("Batch d'entrée :", batch.shape)
    print("Reconstruction :", reconstruction.shape)

    mse = tf.keras.losses.MeanSquaredError()
    print("MSE (poids aléatoires, avant tout entraînement) :", float(mse(batch, reconstruction)))

In [ ]:
# Aperçu visuel : image d'entrée vs reconstruction (avant entraînement — reconstruction non informative à ce stade)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(batch[i].numpy())
    axes[0, i].set_title("Entrée")
    axes[0, i].axis("off")
    axes[1, i].imshow(reconstruction[i].numpy())
    axes[1, i].set_title("Reconstruction\n(avant entraînement)")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## 10. Deux algorithmes à comparer : perte MSE vs perte SSIM

Jusqu'ici, SSIM n'était qu'une **métrique suivie** à côté de la loss MSE (un seul modèle). Pour comprendre l'écart avant/après entraînement propre à chaque critère d'optimisation, on entraîne maintenant **deux modèles indépendants**, de même architecture (`build_autoencoder`), mais chacun optimisé sur sa propre perte :

- **Algorithme A** : `loss = MSE` (la métrique SSIM reste suivie, pour comparaison).
- **Algorithme B** : `loss = 1 - SSIM` (Keras n'a pas de loss SSIM native, on la définit explicitement ; la métrique MSE reste suivie, pour comparaison).

Les deux modèles partent de poids aléatoires indépendants, et sont évalués sur le **même batch de validation fixe** (`x_val_fixed`) avant et après entraînement — condition nécessaire pour que la comparaison "avant/après" soit valide pour chaque algorithme.

In [ ]:
def ssim_metric(y_true, y_pred):
    """SSIM moyen sur le batch, en métrique de suivi (max_val=1.0 car images normalisées [0,1])."""
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
ssim_metric.__name__ = "ssim"

def ssim_loss(y_true, y_pred):
    """Perte SSIM : 1 - SSIM, minimisée quand la reconstruction est structurellement fidèle (SSIM -> 1)."""
    return 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

def mse_metric(y_true, y_pred):
    """MSE en métrique de suivi (pour le modèle entraîné avec la loss SSIM)."""
    return tf.reduce_mean(tf.square(y_true - y_pred))
mse_metric.__name__ = "mse"

train_ds_xy = train_ds.map(lambda x: (x, x))
val_ds_xy = val_ds.map(lambda x: (x, x))

# Deux instances indépendantes, mêmes hyperparamètres d'architecture
model_mse = build_autoencoder(IMG_SIZE)
model_ssim = build_autoencoder(IMG_SIZE)

# Même batch de validation fixe pour une comparaison avant/après valide sur les deux algorithmes
for batch in val_ds_xy.take(1):
    x_val_fixed, _ = batch

recon_mse_before = model_mse(x_val_fixed)
recon_ssim_before = model_ssim(x_val_fixed)

print("Reconstructions 'avant entraînement' capturées pour les deux modèles (même batch de validation, poids aléatoires).")

## 11. Suivi MLflow

Même convention que le reste du projet (SQLite locale, `mlflow/mlflow.db`). **Un run MLflow distinct par algorithme** (`conv_autoencoder_mse` et `conv_autoencoder_ssim`), pour comparer les deux dans l'UI MLflow (`mlflow ui --backend-store-uri sqlite:///mlflow/mlflow.db`) :
- **Params** : `img_size`, `batch_size`, `epochs`, `early_stopping_patience`, `base_filters`, `optimizer`, `loss` (différent selon l'algorithme).
- **Metrics** par epoch : la loss native de l'algorithme (train/val) + la métrique croisée de l'autre critère (train/val), pour comparaison directe.
- **Artifacts** : courbes d'apprentissage et reconstructions avant/après, par algorithme.

In [ ]:
MLFLOW_TRACKING_URI = "sqlite:///mlflow/mlflow.db"  # <- adapter si besoin (cohérent avec le reste du projet)
EXPERIMENT_NAME = "bottle_autoencoder"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

EPOCHS = 30          # nombre maximum d'epochs
PATIENCE = 5         # arrêt anticipé si val_loss ne s'améliore plus pendant PATIENCE epochs

## 12. Algorithme A — Entraînement (perte MSE)

`loss="mse"`, SSIM suivie comme métrique croisée.

In [ ]:
early_stopping_mse = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=PATIENCE, restore_best_weights=True,
)

model_mse.compile(optimizer="adam", loss="mse", metrics=[ssim_metric])

with mlflow.start_run(run_name="conv_autoencoder_mse") as run_mse:
    mlflow.log_params({
        "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "early_stopping_patience": PATIENCE, "base_filters": 32,
        "optimizer": "adam", "loss": "mse",
    })

    history_mse = model_mse.fit(
        train_ds_xy, validation_data=val_ds_xy,
        epochs=EPOCHS, callbacks=[early_stopping_mse], verbose=2,
    )

    epochs_trained_mse = len(history_mse.history["loss"])
    mlflow.log_param("epochs_trained", epochs_trained_mse)

    for epoch in range(epochs_trained_mse):
        mlflow.log_metrics({
            "train_loss": history_mse.history["loss"][epoch],
            "val_loss": history_mse.history["val_loss"][epoch],
            "train_ssim": history_mse.history["ssim"][epoch],
            "val_ssim": history_mse.history["val_ssim"][epoch],
        }, step=epoch)

    run_id_mse = run_mse.info.run_id

print(f"Run MLflow (MSE) terminé : {run_id_mse} — {epochs_trained_mse}/{EPOCHS} epochs effectuées.")

## 13. Algorithme A — Courbes d'apprentissage (MSE)

- **Loss (MSE)** : doit décroître sur train et validation.
- **SSIM (métrique croisée)** : doit croître vers 1.0 — permet de voir si optimiser directement le MSE améliore aussi la similarité structurelle, ou si les deux critères divergent.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history_mse.history["loss"], label="train")
axes[0].plot(history_mse.history["val_loss"], label="validation")
axes[0].set_title("Loss (MSE) — Algorithme MSE")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_mse.history["ssim"], label="train")
axes[1].plot(history_mse.history["val_ssim"], label="validation")
axes[1].set_title("SSIM (métrique croisée) — Algorithme MSE")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("SSIM"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()

with mlflow.start_run(run_id=run_id_mse):
    mlflow.log_figure(fig, f"curves_{run_id_mse}.png")

plt.show()

## 14. Algorithme A — Reconstruction avant / après entraînement

Même batch de validation (`x_val_fixed`) que la capture "avant entraînement" de la section 10 — comparaison directe de l'écart apporté par l'entraînement sur la perte MSE.

In [ ]:
recon_mse_after = model_mse(x_val_fixed)

n_show = min(4, x_val_fixed.shape[0])
fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))
for i in range(n_show):
    axes[0, i].imshow(x_val_fixed[i].numpy()); axes[0, i].set_title("Original"); axes[0, i].axis("off")
    axes[1, i].imshow(recon_mse_before[i].numpy()); axes[1, i].set_title("Avant entraînement"); axes[1, i].axis("off")
    axes[2, i].imshow(recon_mse_after[i].numpy()); axes[2, i].set_title("Après entraînement\n(perte MSE)"); axes[2, i].axis("off")
plt.suptitle("Algorithme MSE — écart avant / après entraînement")
plt.tight_layout()

with mlflow.start_run(run_id=run_id_mse):
    mlflow.log_figure(fig, f"reconstructions_before_after_{run_id_mse}.png")

plt.show()

## 15. Algorithme B — Entraînement (perte SSIM)

`loss=ssim_loss` (1 - SSIM), MSE suivie comme métrique croisée. Même `EPOCHS`/`PATIENCE` que l'algorithme A, pour une comparaison équitable.

In [ ]:
early_stopping_ssim = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=PATIENCE, restore_best_weights=True,
)

model_ssim.compile(optimizer="adam", loss=ssim_loss, metrics=[mse_metric])

with mlflow.start_run(run_name="conv_autoencoder_ssim") as run_ssim:
    mlflow.log_params({
        "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "early_stopping_patience": PATIENCE, "base_filters": 32,
        "optimizer": "adam", "loss": "ssim_loss (1 - SSIM)",
    })

    history_ssim = model_ssim.fit(
        train_ds_xy, validation_data=val_ds_xy,
        epochs=EPOCHS, callbacks=[early_stopping_ssim], verbose=2,
    )

    epochs_trained_ssim = len(history_ssim.history["loss"])
    mlflow.log_param("epochs_trained", epochs_trained_ssim)

    for epoch in range(epochs_trained_ssim):
        mlflow.log_metrics({
            "train_loss": history_ssim.history["loss"][epoch],
            "val_loss": history_ssim.history["val_loss"][epoch],
            "train_mse": history_ssim.history["mse"][epoch],
            "val_mse": history_ssim.history["val_mse"][epoch],
        }, step=epoch)

    run_id_ssim = run_ssim.info.run_id

print(f"Run MLflow (SSIM) terminé : {run_id_ssim} — {epochs_trained_ssim}/{EPOCHS} epochs effectuées.")

## 16. Algorithme B — Courbes d'apprentissage (SSIM)

- **Loss (1 - SSIM)** : doit décroître (donc SSIM croît vers 1.0) sur train et validation.
- **MSE (métrique croisée)** : à surveiller — optimiser SSIM ne garantit pas de minimiser le MSE, les deux courbes peuvent diverger, c'est justement ce que cette comparaison doit révéler.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history_ssim.history["loss"], label="train")
axes[0].plot(history_ssim.history["val_loss"], label="validation")
axes[0].set_title("Loss (1 - SSIM) — Algorithme SSIM")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("1 - SSIM"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_ssim.history["mse"], label="train")
axes[1].plot(history_ssim.history["val_mse"], label="validation")
axes[1].set_title("MSE (métrique croisée) — Algorithme SSIM")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MSE"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()

with mlflow.start_run(run_id=run_id_ssim):
    mlflow.log_figure(fig, f"curves_{run_id_ssim}.png")

plt.show()

## 17. Algorithme B — Reconstruction avant / après entraînement

Même batch de validation fixe, même mise en page que pour l'algorithme A (section 14) — comparaison directe possible entre les deux algorithmes (section 18).

In [ ]:
recon_ssim_after = model_ssim(x_val_fixed)

n_show = min(4, x_val_fixed.shape[0])
fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))
for i in range(n_show):
    axes[0, i].imshow(x_val_fixed[i].numpy()); axes[0, i].set_title("Original"); axes[0, i].axis("off")
    axes[1, i].imshow(recon_ssim_before[i].numpy()); axes[1, i].set_title("Avant entraînement"); axes[1, i].axis("off")
    axes[2, i].imshow(recon_ssim_after[i].numpy()); axes[2, i].set_title("Après entraînement\n(perte SSIM)"); axes[2, i].axis("off")
plt.suptitle("Algorithme SSIM — écart avant / après entraînement")
plt.tight_layout()

with mlflow.start_run(run_id=run_id_ssim):
    mlflow.log_figure(fig, f"reconstructions_before_after_{run_id_ssim}.png")

plt.show()

## 18. Comparaison des deux algorithmes

Les deux modèles reconstruisent les **mêmes images de validation** — comparaison directe de l'effet du choix de la loss sur la qualité perçue de la reconstruction, et sur les métriques finales.

In [ ]:
n_show = min(4, x_val_fixed.shape[0])
fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))
for i in range(n_show):
    axes[0, i].imshow(x_val_fixed[i].numpy()); axes[0, i].set_title("Original"); axes[0, i].axis("off")
    axes[1, i].imshow(recon_mse_after[i].numpy()); axes[1, i].set_title("Après entraînement\n(MSE)"); axes[1, i].axis("off")
    axes[2, i].imshow(recon_ssim_after[i].numpy()); axes[2, i].set_title("Après entraînement\n(SSIM)"); axes[2, i].axis("off")
plt.suptitle("Comparaison des deux algorithmes sur les mêmes images de validation")
plt.tight_layout()
plt.show()

print("Résumé final :")
print(f"  MSE  — {epochs_trained_mse} epochs — val_loss(MSE)={history_mse.history['val_loss'][-1]:.5f} — val_ssim={history_mse.history['val_ssim'][-1]:.4f}")
print(f"  SSIM — {epochs_trained_ssim} epochs — val_loss(1-SSIM)={history_ssim.history['val_loss'][-1]:.5f} — val_mse={history_ssim.history['val_mse'][-1]:.5f}")

## 19. Score d'anomalie et calibration du seuil (comparaison MSE vs SSIM)

Même principe que précédemment (score = erreur de reconstruction MSE par image, seuil calibré uniquement sur `val_ds`), appliqué **séparément aux deux modèles entraînés**, pour voir si le choix de la loss d'entraînement change la capacité de discrimination saines/défauts du score d'anomalie.

In [ ]:
def reconstruction_errors(dataset, model):
    """Erreur de reconstruction (MSE) par image — un score par image, pas par batch."""
    errors = []
    for batch in dataset:
        recon = model(batch)
        err = tf.reduce_mean(tf.square(batch - recon), axis=[1, 2, 3])
        errors.append(err.numpy())
    return np.concatenate(errors)

THRESHOLD_PERCENTILE = 95
models = {"MSE": (model_mse, run_id_mse), "SSIM": (model_ssim, run_id_ssim)}
scores, thresholds, recalls, fprs = {}, {}, {}, {}

for name, (model, rid) in models.items():
    val_scores = reconstruction_errors(val_ds, model)
    test_good_scores = reconstruction_errors(test_good_ds, model)
    test_defect_scores = np.concatenate([reconstruction_errors(ds, model) for ds in test_defect_ds.values()])

    threshold = np.percentile(val_scores, THRESHOLD_PERCENTILE)
    recall = (test_defect_scores > threshold).mean()
    fpr = (test_good_scores > threshold).mean()

    scores[name] = {"val": val_scores, "good": test_good_scores, "defect": test_defect_scores}
    thresholds[name] = threshold
    recalls[name] = recall
    fprs[name] = fpr

    with mlflow.start_run(run_id=rid):
        mlflow.log_metrics({"threshold_p95": threshold, "recall_p95": recall, "fpr_p95": fpr})

    print(f"[{name}] seuil(p{THRESHOLD_PERCENTILE})={threshold:.5f}  rappel={recall:.2%}  faux positifs={fpr:.2%}")

### Rendu visuel — histogrammes saines vs défauts, un panneau par algorithme

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, name in zip(axes, ["MSE", "SSIM"]):
    good, defect, threshold = scores[name]["good"], scores[name]["defect"], thresholds[name]
    bins = np.linspace(min(good.min(), defect.min()), max(good.max(), defect.max()), 25)
    ax.hist(good, bins=bins, alpha=0.6, label="test/good (saines)", color="#55A868")
    ax.hist(defect, bins=bins, alpha=0.6, label="test/<defect> (défauts)", color="#C44E52")
    ax.axvline(threshold, color="black", linestyle="--", label=f"seuil (p{THRESHOLD_PERCENTILE})")
    ax.set_title(f"Algorithme {name} — rappel={recalls[name]:.0%}, FP={fprs[name]:.0%}")
    ax.set_xlabel("Erreur de reconstruction (MSE par image)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Nombre d'images")
plt.tight_layout()
plt.show()

### Abaisser le seuil augmente le rappel — comparaison des deux algorithmes

Rendu distinct : rappel et taux de faux positifs en fonction du seuil (balayage de percentiles), une courbe par algorithme — permet de voir directement si un algorithme offre un meilleur compromis rappel/faux-positifs que l'autre, à n'importe quel seuil.

In [ ]:
percentiles = np.arange(50, 100, 2)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name, color in [("MSE", "#4C72B0"), ("SSIM", "#DD8452")]:
    val_scores, good, defect = scores[name]["val"], scores[name]["good"], scores[name]["defect"]
    recall_curve = [(defect > np.percentile(val_scores, p)).mean() for p in percentiles]
    fpr_curve = [(good > np.percentile(val_scores, p)).mean() for p in percentiles]
    axes[0].plot(percentiles, recall_curve, marker="o", label=f"{name}", color=color)
    axes[1].plot(percentiles, fpr_curve, marker="o", label=f"{name}", color=color)

for ax, title in zip(axes, ["Rappel vs seuil", "Faux positifs vs seuil"]):
    ax.axvline(THRESHOLD_PERCENTILE, color="black", linestyle="--", alpha=0.4, label=f"seuil par défaut (p{THRESHOLD_PERCENTILE})")
    ax.set_xlabel("Percentile utilisé pour le seuil (calibré sur les saines de validation)")
    ax.set_title(title)
    ax.invert_xaxis()
    ax.legend()
axes[0].set_ylabel("Rappel")
axes[1].set_ylabel("Taux de faux positifs")
plt.tight_layout()

with mlflow.start_run(run_id=run_id_mse):
    mlflow.log_figure(fig, f"recall_vs_threshold_comparison_{run_id_mse}.png")

plt.show()

**Lecture** : pour les deux algorithmes, le rappel augmente quand on abaisse le seuil, au prix de plus de faux positifs — pas de seuil "gratuit". Si une courbe domine l'autre (meilleur rappel pour un même taux de faux positifs), l'algorithme correspondant offre un meilleur score d'anomalie intrinsèque. Le choix final du seuil dépend ensuite du coût métier relatif d'un défaut manqué vs d'une fausse alerte (cf. `etude_desequilibre_classe.md`).

## 20. Prochaine étape

Le seuil calibré ici (p95 des scores de validation) est un point de départ raisonnable, pas une valeur définitive, pour l'un ou l'autre algorithme. Pistes de suite :
- Choisir l'algorithme (MSE ou SSIM) qui domine sur la courbe rappel/faux-positifs (section 19), plutôt que d'en présupposer un a priori.
- Ajuster le percentile de calibration selon le coût métier réel d'un faux positif vs d'un faux négatif.
- Évaluer avec des métriques agrégées indépendantes du seuil (AUC-ROC, AUC-PR) plutôt qu'à un seul point de fonctionnement.
- Passer à un score pixel-level (carte d'erreur de reconstruction par pixel) pour localiser le défaut, en s'appuyant sur `ground_truth/` (IoU, AUC-ROC pixel) — cf. étude de déséquilibre pixel-level en étape 3.11.